# The burnt marker: an orientation-proof reacting closure

A reacting network labels each edge **frozen** (unburnt reactants) or **equilibrium** (burnt products).
Getting that label from the *drawn* edge arrows is fragile: an edge may legitimately carry a negative mass
flow, so a flame edge drawn pointing the wrong way is silently mislabeled.

The fix is a transported **burnt marker** scalar `b`.
Every reacting edge runs a single `EQ_MARKER` closure that blends the frozen and equilibrium states by a
smooth gate of `b`; the flame stamps `b = 1` on whatever edge the flow actually leaves it by (the marker
rides the *signed* mass flow), so the split is correct no matter how the edges were drawn.
The old topology flood-fill survives only as the marker's initial guess -- the transport self-corrects it.

In [ ]:
import os, sys
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go

from nefes.thermo import SpeciesLibrary, Thermo
from nefes.elements import catalog as cat
from nefes.shell.network import Network
from nefes.solver.control import auto_initial_guess
from nefes.thermo.configure import equilibrium
from nefes.thermo.api import EQ_MARKER
from nefes.thermo.edge_state import MARKER_GATE_WIDTH
from nefes.assembly.smooth import marker_gate

MECH = os.path.join(_root, "nefes", "thermo", "data", "h2o2.yaml")
lib = SpeciesLibrary.from_cantera(MECH)
gas = Thermo(lib)
AIR = {"O2": 0.21, "N2": 0.79}
idx = lib.species_index
_Y = np.zeros(lib.n_species); _Y[idx['O2']], _Y[idx['N2']] = 0.21, 0.79; _Y /= _Y.sum()
H_AIR = gas.enthalpy_mass(_Y, 300.0)  # absolute-enthalpy datum for the air feed

## The smooth blend gate

The closure blends `(1 - g)` of the frozen state with `g` of the equilibrium state, where `g = marker_gate(b)`.
The gate is normalized so `g(0) = 0` and `g(1) = 1` *exactly* -- a fresh edge is pure frozen, a burnt edge pure
equilibrium -- while staying smooth (and complex-step-safe) in between, so the coupled Newton solve sees no
branch.
The marker is bimodal at convergence, so the blend is only ever active during the iteration.

In [ ]:
b = np.linspace(-0.3, 1.3, 400)
g = np.array([marker_gate(float(x), MARKER_GATE_WIDTH) for x in b])
fig = go.Figure()
fig.add_trace(go.Scatter(x=b, y=g, mode='lines', line=dict(color='#c0392b', width=3), name='g = marker_gate(b)'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='markers',
                         marker=dict(size=10, color='#2c3e50'), name='converged b in {0, 1}'))
fig.update_layout(template='plotly_white', width=620, height=380,
                  title='Blend gate: 0 = frozen (unburnt), 1 = equilibrium (burnt)',
                  xaxis_title='burnt marker  b', yaxis_title='equilibrium weight  g')
fig.show()

## A marker-gated combustor

Air enters, a fuel stream is injected, a flame sits mid-duct, products leave.
We build it with the plain `Network` API and **no** per-edge closure override, so it is marker-gated
automatically: every edge is `EQ_MARKER` and one extra `burnt` scalar is transported.

In [ ]:
def combustor(edges):
    nodes = [
        cat.total_pressure_inlet(1.2e5, 300.0, composition=AIR, basis='mole', name='air'),
        cat.duct(0.4),
        cat.mass_source(0.006, 300.0, composition={'H2': 1.0}, basis='mole', name='H2'),
        cat.duct(0.4),
        cat.equilibrium_flame(name='flame'),
        cat.duct(0.5),
        cat.mass_flow_outlet(0.406, name='out'),
    ]
    return Network(gas=equilibrium(gas.mech), p_ref=1e5, T_ref=300.0, mdot_ref=0.4, h_ref=H_AIR,
                   nodes=nodes, edges=edges)

# edge 4 is the post-flame (burnt) edge; the rest are the unburnt approach
edges = [(0, 1, 0.01), (1, 2, 0.01), (2, 3, 0.01), (3, 4, 0.01), (4, 5, 0.01), (5, 6, 0.01)]
net = combustor(edges)
prob = net.compile()
print('per-edge model :', ['EQ_MARKER' if m == EQ_MARKER else int(m) for m in prob.edge_model])
print('n_solve        :', prob.n_solve, ' (3 + %d streams + 1 marker)' % prob.n_elem)
print('marker seed    :', prob.marker_seed.tolist(), ' <- the flood-fill, only an initial guess')

sol = net.solve()
print('converged      :', sol.converged, ' in', sol.iterations, 'iterations')

In [ ]:
E = prob.n_edges
b_field = [sol.marker(e) for e in range(E)]
T = sol.field('T')
fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(E)), y=b_field, marker_color='#2980b9', name='burnt marker b',
                     yaxis='y1'))
fig.add_trace(go.Scatter(x=list(range(E)), y=T, mode='lines+markers', line=dict(color='#c0392b', width=3),
                         name='static T [K]', yaxis='y2'))
fig.update_layout(template='plotly_white', width=720, height=400,
                  title='Marker = 1 exactly where the gas is burnt; T jumps with it',
                  xaxis_title='edge',
                  yaxis=dict(title='burnt marker', range=[0, 1.05]),
                  yaxis2=dict(title='T [K]', overlaying='y', side='right', showgrid=False))
fig.show()

## Orientation robustness: the seed is only a guess

The flood-fill that seeds the marker follows the *drawn* arrows, so a backward-drawn flame seeds it wrong.
It does not matter: the marker transport rides the signed mass flow, so it converges to the **same** field
from any starting guess.
Below we scramble the initial marker three different (wrong) ways and recover the same answer each time.

In [ ]:
mr = prob.marker_row
scrambles = {
    'flood-fill seed': prob.marker_seed.tolist(),
    'all fresh (0)':   [0, 0, 0, 0, 0, 0],
    'inverted':        [1, 1, 1, 1, 0, 0],
    'all burnt (1)':   [1, 1, 1, 1, 1, 1],
}
fig = go.Figure()
for name, seed in scrambles.items():
    x0 = auto_initial_guess(prob)
    x0[mr, :] = seed
    r = combustor(edges).solve(x0=x0)
    conv = [round(r.marker(e), 4) for e in range(E)]
    print(f'{name:16s} seed={seed} -> converged marker={conv}  (conv={r.converged})')
    fig.add_trace(go.Scatter(x=list(range(E)), y=conv, mode='lines+markers', name=name))
fig.update_layout(template='plotly_white', width=720, height=380,
                  title='Every scrambled seed converges to the same marker field',
                  xaxis_title='edge', yaxis_title='converged burnt marker')
fig.show()

## Post-processing: marker, species, molar mass, cp

Everything the closure computes is surfaced per edge after the solve (and goes into the YAML output by
default).
The species readout follows the marker: an unburnt edge reports its frozen reactants, a burnt edge its
equilibrium products.

In [ ]:
for e in (3, 4):  # the flame-approach (fresh) edge and the flame-out (burnt) edge
    tag = 'BURNT ' if sol.marker(e) > 0.5 else 'fresh '
    top = sorted(sol.species(e, basis='mole').items(), key=lambda kv: -kv[1])[:4]
    sp = ', '.join(f'{n} {x:.3f}' for n, x in top)
    print(f'edge {e} [{tag}] b={sol.marker(e):.3f}  T={sol.field("T")[e]:7.1f} K  '
          f'W={sol.field("W")[e]*1e3:5.2f} g/mol  cp={sol.field("cp")[e]:6.1f} J/kgK  | {sp}')

## The marker is acoustically passive

At convergence the marker is saturated (`b in {0, 1}`), so the gate is flat there and the marker decouples
from the acoustics -- it is just a convected scalar.
The acoustic transfer matrix across the flame is therefore identical to the one from an explicit hard
`EQ_FROZEN` / `EQ_KERNEL` closure (no marker).

In [ ]:
from nefes.thermo.api import EQ_FROZEN, EQ_KERNEL
from nefes.perturbation.response.response import perturbation_response

freqs = np.linspace(50.0, 800.0, 60)
x_marker = sol.x

# the same geometry with an explicit hard closure (frozen approach, equilibrium downstream)
hard = cat.build_problem(equilibrium(gas.mech), net._elements, edges, mdot_ref=0.4, p_ref=1e5, h_ref=H_AIR,
                         edge_models=[EQ_FROZEN, EQ_FROZEN, EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL])
from nefes.solver import solve as _solve
x_hard = _solve(hard).x

rm = perturbation_response(prob, x_marker, freqs, excite=('acoustic', 'entropy'))
rh = perturbation_response(hard, x_hard, freqs, excite=('acoustic', 'entropy'))
Tm = rm.transfer_matrix(0, 4)  # inlet edge -> burnt edge, across the flame
Th = rh.transfer_matrix(0, 4)

fig = go.Figure()
fig.add_trace(go.Scatter(x=freqs, y=np.abs(Tm[:, 0, 0]), mode='lines', line=dict(color='#c0392b', width=4),
                         opacity=0.5, name='marker-gated  |T(f→f)|'))
fig.add_trace(go.Scatter(x=freqs, y=np.abs(Th[:, 0, 0]), mode='lines', line=dict(color='#2c3e50', dash='dot'),
                         name='hard closure  |T(f→f)|'))
fig.update_layout(template='plotly_white', width=720, height=380,
                  title='Marker-gated vs hard-closure acoustic transfer across the flame (overlaid)',
                  xaxis_title='frequency [Hz]', yaxis_title='|T| (downstream acoustic)')
fig.show()
print('max |T_marker - T_hard| over the sweep:', float(np.max(np.abs(Tm - Th))))

## User-settable inflow marker: feeding already-burnt gas

An inlet (or source) normally enters *fresh* (`marker = 0`, the default).
Setting `marker = 1` injects **already-burnt** gas -- e.g. exhaust-gas recirculation as a feed -- which forces
the equilibrium closure on the inflow edge itself, so a premixed feed burns right at the inlet instead of
staying frozen until the downstream flame.
(The marker only applies on a marker-gated network -- a reacting net with an equilibrium flame and no explicit
per-edge closure -- so a non-zero value elsewhere is rejected at build time.)

In [ ]:
PREMIX = {'H2': 0.296, 'O2': 0.148, 'N2': 0.556}  # stoichiometric H2-air (by mole)
_Yp = np.zeros(lib.n_species)
for _sp, _x in PREMIX.items():
    _Yp[idx[_sp]] = _x
_Yp /= _Yp.sum()
H_PREMIX = gas.enthalpy_mass(_Yp, 300.0)  # absolute-enthalpy datum for the premix feed

def premixed(marker):
    nodes = [
        cat.total_pressure_inlet(1.2e5, 300.0, composition=PREMIX, basis='mole', marker=marker, name='feed'),
        cat.equilibrium_flame(name='flame'),
        cat.pressure_outlet(1.0e5, 300.0, composition=PREMIX, basis='mole', name='out'),
    ]
    net = Network(gas=equilibrium(gas.mech), p_ref=1e5, T_ref=300.0, mdot_ref=0.4, h_ref=H_PREMIX,
                  nodes=nodes, edges=[(0, 1, 0.01), (1, 2, 0.01)])
    return net.solve()

for m in (0.0, 1.0):
    s = premixed(m)
    T = s.field('T')
    tag = 'fresh feed: frozen until the flame' if m == 0 else 'burnt feed: equilibrium at the inlet'
    print(f'marker={m:.0f}:  inflow-edge T = {T[0]:7.1f} K,  outlet T = {T[1]:7.1f} K   [{tag}]')

## Export for FNetLibUI

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in FNetLibUI.

In [ ]:
import os, tempfile

network, sol = sol.network, sol  # the primary Network and its mean-flow Solution
_out = os.path.join(tempfile.mkdtemp(), "burnt_marker.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)